<a href="https://colab.research.google.com/github/theelderemo/wan2.2-google-colab/blob/main/wan2_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/Wan-Video/Wan2.2.git

In [ ]:
%cd Wan2.2

In [ ]:
!pip install -r requirements.txt

In [ ]:
!pip install .

In [ ]:
!pip install flash-attn --no-build-isolation

In [ ]:
!pip install "huggingface_hub[cli]"

In [ ]:
!huggingface-cli download Wan-AI/Wan2.2-I2V-A14B --local-dir ./Wan2.2-I2V-A14B

In [ ]:
# @title 1. Upload Input Image
import os
from google.colab import files

# Clean up previous uploads to avoid confusion
if 'input_image_path' in locals():
    print(f"Previous image was: {input_image_path}")
else:
    input_image_path = ""

uploaded = files.upload()

# Get the filename of the uploaded file
if uploaded:
    input_image_path = list(uploaded.keys())[0]
    print(f"\nSUCCESS: Image set to '{input_image_path}'")
else:
    print("\nNo file uploaded.")

In [ ]:
# @title 2. Generation Settings
# @markdown ### 📝 Prompting
prompt = "prompt here" # @param {type:"string"}

# @markdown ### ⚙️ Model Config
task = "i2v-A14B" # @param ["i2v-A14B", "i2v-1.3B"]
resolution = "1280*720" # @param ["1280*720", "832*480", "1024*1024"]
checkpoint_dir = "./Wan2.2-I2V-A14B" # @param {type:"string"}

# @markdown ### 🚀 Optimization Flags
offload_model = True # @param {type:"boolean"}
use_t5_cpu = True # @param {type:"boolean"}
convert_model_dtype = True # @param {type:"boolean"}

# Constructing the argument string based on checkboxes
args = ""
if offload_model:
    args += "--offload_model True "
if use_t5_cpu:
    args += "--t5_cpu "
if convert_model_dtype:
    args += "--convert_model_dtype "

print("Configuration updated.")

In [ ]:
# @title 3. Generate Video
import torch
import sys
import os

# Install missing dependency 'decord' which caused the error
!pip install decord

# 1. Clear VRAM before starting
torch.cuda.empty_cache()

# 2. Check if image exists
if not os.path.exists(input_image_path):
    print(f"ERROR: Could not find image '{input_image_path}'. Please run Cell 1 again.")
else:
    print(f"Processing image: {input_image_path}...")
    print(f"Prompt: {prompt}")

    # 3. Execute Command
    # We use f-strings to insert the variables from Cell 2
    !python generate.py \
        --task {task} \
        --size {resolution} \
        --ckpt_dir {checkpoint_dir} \
        --image "{input_image_path}" \
        --prompt "{prompt}" \
        {args}

In [ ]:
import glob
from IPython.display import Video

# Find the most recently generated mp4 file
mp4_files = glob.glob("*.mp4")
if mp4_files:
    latest_file = max(mp4_files, key=os.path.getctime)
    print(f"Displaying: {latest_file}")
    display(Video(latest_file, embed=True))
else:
    print("No video found.")